In [16]:
import pathlib
import sys
import os


def find_project_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    return start


ROOT_DIR = find_project_root(pathlib.Path.cwd().resolve())
sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.data_loader import load_data
from src.analytics.batting import player_milestones

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

matches, deliveries = load_data()
print("✅ Batting data loaded successfully!")
print(f"Matches: {len(matches)}, Deliveries: {len(deliveries)}")


✅ Batting data loaded successfully!
Matches: 1169, Deliveries: 278205


## 1. Top Run Scorers


In [17]:
# Top 15 run scorers
top_batsmen = deliveries.groupby("batsman")["batsman_runs"].sum().sort_values(ascending=False).head(15)

fig = px.bar(
    x=top_batsmen.values,
    y=top_batsmen.index,
    orientation="h",
    title="Top 15 IPL Run Scorers (2008-2025)",
    labels={"x": "Runs", "y": "Batsman"},
    color=top_batsmen.values,
    color_continuous_scale="Viridis"
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("📊 Top 15 Run Scorers:")
print(top_batsmen)


📊 Top 15 Run Scorers:
batsman
V Kohli           8671
RG Sharma         7048
S Dhawan          6769
DA Warner         6567
SK Raina          5536
MS Dhoni          5439
KL Rahul          5235
AB de Villiers    5181
AM Rahane         5032
CH Gayle          4997
RV Uthappa        4954
KD Karthik        4843
F du Plessis      4773
SV Samson         4704
AT Rayudu         4348
Name: batsman_runs, dtype: int64


## 2. Batsman Strike Rate Analysis


In [18]:
# Calculate strike rate (minimum 300 balls faced)
batsman_balls = deliveries.groupby("batsman").size()
batsman_runs = deliveries.groupby("batsman")["batsman_runs"].sum()
strike_rate = (batsman_runs / batsman_balls * 100).sort_values(ascending=False)

# Filter batsmen with at least 300 balls
sr_filtered = strike_rate[batsman_balls >= 300].head(15)

fig = px.bar(
    x=sr_filtered.values,
    y=sr_filtered.index,
    orientation="h",
    title="Top 15 Strike Rates (Min 300 balls)",
    labels={"x": "Strike Rate", "y": "Batsman"},
    color=sr_filtered.values,
    color_continuous_scale="RdYlGn"
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("🏏 Top Strike Rates (Min 300 balls):")
print(sr_filtered)


🏏 Top Strike Rates (Min 300 balls):
batsman
Priyansh Arya      175.806452
PD Salt            169.502408
AD Russell         163.284133
H Klaasen          163.175303
TM Head            162.322946
TH David           161.450382
T Stubbs           160.859729
N Pooran           160.013957
Abhishek Sharma    155.612682
SP Narine          155.594406
Shashank Singh     151.568627
WG Jacks           151.307190
Rashid Khan        151.162791
JM Sharma          151.067073
LS Livingstone     150.357654
dtype: float64


## 3. Runs Scored by Season


In [19]:
# Total runs by season
season_lookup = matches[["matchId", "season"]].drop_duplicates()
deliveries_with_season = deliveries.merge(season_lookup, on="matchId", how="left")
runs_by_season = (
    deliveries_with_season.dropna(subset=["season"])
    .groupby("season")["batsman_runs"]
    .sum()
    .sort_index()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=runs_by_season.index,
    y=runs_by_season.values,
    mode="lines+markers",
    name="Total Runs",
    line=dict(color="#1f77b4", width=3),
    marker=dict(size=8)
))

fig.update_layout(
    title="Total Runs Scored by Season",
    xaxis_title="Season",
    yaxis_title="Runs",
    height=500,
    hovermode="x unified"
)
fig.show()

print("📈 Total Runs by Season:")
print(runs_by_season)


📈 Total Runs by Season:
season
2007/08    16809
2009       15376
2009/10    17754
2011       19928
2012       21323
2013       21487
2014       17943
2015       17427
2016       17962
2017       17920
2018       19098
2019       18607
2020/21    18566
2021       17727
2022       23052
2023       24428
2024       24657
2025       25309
Name: batsman_runs, dtype: int64


## 4. Boundary Analysis


In [20]:
# Boundary analysis (4s and 6s)
fours = len(deliveries[deliveries["batsman_runs"] == 4])
sixes = len(deliveries[deliveries["batsman_runs"] == 6])
singles = len(deliveries[deliveries["batsman_runs"] == 1])
dots = len(deliveries[deliveries["batsman_runs"] == 0])
twos = len(deliveries[deliveries["batsman_runs"] == 2])

boundary_data = {
    "Type": ["Dot Balls", "Singles", "Twos", "4s", "6s"],
    "Count": [dots, singles, twos, fours, sixes]
}

fig = px.pie(
    values=boundary_data["Count"],
    names=boundary_data["Type"],
    title="Distribution of Ball Types (All Deliveries)",
    color_discrete_map={"Dot Balls": "#FF6B6B", "Singles": "#4ECDC4", "Twos": "#45B7D1", "4s": "#FFA07A", "6s": "#98D8C8"}
)
fig.show()

print("🎯 Ball Type Distribution:")
for i, ball_type in enumerate(boundary_data["Type"]):
    print(f"{ball_type}: {boundary_data["Count"][i]} ({boundary_data["Count"][i]/sum(boundary_data["Count"])*100:.2f}%)")


🎯 Ball Type Distribution:
Dot Balls: 110250 (39.75%)
Singles: 103187 (37.21%)
Twos: 17424 (6.28%)
4s: 32113 (11.58%)
6s: 14353 (5.18%)


## 5. Top Sixes Hitters


In [21]:
# Top 6 hitters
sixes_by_batsman = deliveries[deliveries["batsman_runs"] == 6].groupby("batsman").size().sort_values(ascending=False).head(15)

fig = px.bar(
    x=sixes_by_batsman.values,
    y=sixes_by_batsman.index,
    orientation="h",
    title="Top 15 Sixes Hitters in IPL",
    labels={"x": "Number of Sixes", "y": "Batsman"},
    color=sixes_by_batsman.values,
    color_continuous_scale="Reds"
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("💥 Top 15 Six Hitters:")
print(sixes_by_batsman)


💥 Top 15 Six Hitters:
batsman
CH Gayle          359
RG Sharma         303
V Kohli           292
MS Dhoni          264
AB de Villiers    253
DA Warner         236
KA Pollard        224
AD Russell        223
SV Samson         219
KL Rahul          208
SK Raina          204
SR Watson         190
JC Buttler        185
RV Uthappa        182
F du Plessis      174
dtype: int64


## 6. Century and Half-Century Analysis


In [22]:
# Centuries and half-centuries
centuries = []
half_centuries = []

for batsman in deliveries["batsman"].unique():
    milestones = player_milestones(deliveries, batsman)
    if milestones["100s"] > 0:
        centuries.append((batsman, milestones["100s"]))
    if milestones["50s"] > 0:
        half_centuries.append((batsman, milestones["50s"]))

centuries_df = pd.DataFrame(centuries, columns=["Batsman", "Centuries"]).sort_values("Centuries", ascending=False).head(10)
half_centuries_df = pd.DataFrame(half_centuries, columns=["Batsman", "Half-Centuries"]).sort_values("Half-Centuries", ascending=False).head(10)

fig1 = px.bar(centuries_df, x="Batsman", y="Centuries", title="Top 10 Century Makers", color="Centuries", color_continuous_scale="Blues")
fig1.update_layout(height=400)
fig1.show()

fig2 = px.bar(half_centuries_df, x="Batsman", y="Half-Centuries", title="Top 10 Half-Century Makers", color="Half-Centuries", color_continuous_scale="Greens")
fig2.update_layout(height=400)
fig2.show()

print("🏆 Top Century Makers:")
print(centuries_df)
print("\n📊 Top Half-Century Makers:")
print(half_centuries_df)


🏆 Top Century Makers:
           Batsman  Centuries
1          V Kohli          8
35      JC Buttler          7
20        CH Gayle          6
32        KL Rahul          5
23       DA Warner          4
5        SR Watson          4
42    Shubman Gill          4
31       SV Samson          3
17  AB de Villiers          3
0      BB McCullum          2

📊 Top Half-Century Makers:
            Batsman  Half-Centuries
5           V Kohli              64
75        DA Warner              62
25         S Dhawan              51
34        RG Sharma              47
54   AB de Villiers              41
119        KL Rahul              40
13         SK Raina              39
101    F du Plessis              39
23        G Gambhir              36
42        AM Rahane              33
